# Chapter 3 / Paper 2
## Notebook 3: Empirical Analysis

This public notebook documents the empirical workflow for **From Consumer Mobility to Localized Sales Performance: Place-Based Digital Sentiment and the Local Market Conversion Gap**. It integrates the tract-level outputs prepared in Notebooks 1 and 2, expands GEWI coverage under the documented spatial rule, estimates the spatial models, and reproduces the sensitivity and flexible-model diagnostics reported in Chapter 3 and Appendix B.

> **Interpretation boundary.** The study is observational. Consumer mobility represents potential market exposure, GEWI represents one place-based informational condition, and localized sales represent an aggregated market outcome. None of the models identifies individual conversion or a causal effect of mobility or digital sentiment on sales.

### Public-release scope

The notebook contains no empirical observations, raw messages, device histories, individual trajectories, transaction records, credentials, company identifiers, private file paths, or embedded outputs. Full execution requires authorized access to the four tract-level inputs described below. The code writes any locally generated tables and figures only under `data/private/analysis/`, which must remain excluded from version control.

The numerical values displayed in the next section are **reference results already reported in the dissertation**, not fabricated notebook outputs. Authorized users can rerun the workflow and compare the recalculated estimates with these public reference values.

### Reference Results Reported in the Dissertation

| Result | Dissertation value |
|---|---:|
| National mobility-covered universe | 436,868 census tracts |
| Integrated analytical sample | 726 census tracts |
| Directly observed GEWI subset | 96 census tracts |
| Moran's I for log-transformed localized sales | approximately 0.14, p < .001 |
| GEWI leave-one-out reconstruction | R2 = 0.794 |
| Spatial Lag Model mobility elasticity | 0.3275 (SE = 0.0889; p = .0002) |
| Spatial Error Model mobility elasticity | 0.3657 (SE = 0.0909; p = .0001) |
| Mobility x baseline GEWI | 0.127 (p = .0034) |
| Mobility x positive GEWI | 0.072 (p = .012) |
| Mobility x negative GEWI | -0.183 (p = .004) |
| Random Forest diagnostic | test R2 = 0.024; RMSE = 2.215 |
| XGBoost diagnostic | test R2 = -0.058; RMSE = 2.306 |

The polarity-specific difference is descriptive: the dissertation compares the signs and absolute magnitudes of the two interaction estimates and does not report a separate statistical test of their difference.

## 1. Environment and Project-Relative Paths

Install the packages from `chapter_3_paper_2/environment/requirements.txt`. The flexible-partitioning cell uses `econml==0.15.1`; its outputs are treated only as conditional-association diagnostics.

In [ ]:
from pathlib import Path
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

from esda.moran import Moran
from libpysal.weights import KNN
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from spreg import ML_Error, ML_Lag
from xgboost import XGBRegressor

RANDOM_STATE = 42
pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
def find_repository_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "chapter_3_paper_2").is_dir():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from the dissertation repository or one of its subdirectories."
    )


REPOSITORY_ROOT = find_repository_root()
CHAPTER_DIR = REPOSITORY_ROOT / "chapter_3_paper_2"
PRIVATE_DATA_DIR = CHAPTER_DIR / "data" / "private"
ANALYSIS_DIR = PRIVATE_DATA_DIR / "analysis"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"

for directory in [ANALYSIS_DIR, FIGURE_DIR, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INPUTS = {
    "mobility": PRIVATE_DATA_DIR / "mobility" / "mobility_by_tract_aug2024.csv.gz",
    "sales": PRIVATE_DATA_DIR / "sales" / "sales_by_tract_aug2024.csv.gz",
    "gewi": PRIVATE_DATA_DIR / "gewi" / "gewi_observed_by_tract.csv.gz",
    "tracts": PRIVATE_DATA_DIR / "census_tracts" / "census_tracts.gpkg",
}

FOCAL_MOBILITY = "total_unique_visitors"
DIRECT_MESSAGE_THRESHOLD = 5
IDW_RADIUS_METERS = 2_000
IDW_POWER = 1.0
K_NEIGHBORS = 8
PROJECTED_CRS = "EPSG:5880"  # SIRGAS 2000 / Brazil Polyconic
CENSUS_LAYER = None  # Set only when the GeoPackage contains multiple layers.
STRICT_REPLICATION = True

## 2. Data Contract and Canonical Variables

The public workflow begins from tract-level files rather than raw restricted records. Notebooks 1 and 2 document how the mobility and GEWI files are constructed.

Consumer mobility and localized sales share the August 2024 analytical window. GEWI is constructed from historically accumulated geolocated discourse posted between 2010 and July 12, 2023, so it is not interpreted as contemporaneous monthly sentiment.

| Canonical variable | Source | Meaning in this study |
|---|---|---|
| `tract_id` | all inputs | 15-digit census-tract identifier |
| `total_visits` | mobility | aggregated visits in August 2024 |
| `total_unique_visitors` | mobility | distinct aggregated visitors in August 2024 |
| `visit_frequency` | mobility | visits divided by unique visitors |
| `total_repeat_visitors` | mobility | aggregated returning visitors |
| `total_new_visitors` | mobility | aggregated new visitors |
| `avg_dwell_time_mins` | mobility | average visit duration in minutes |
| `total_sales` | sales | downstream transaction value aggregated by tract in August 2024 |
| `message_count` | GEWI | relevant geolocated messages used in direct measurement |
| `gewi_v1` | GEWI | mean polarity multiplied by log message volume |
| `gewi_positive` | GEWI | positive-polarity GEWI component |
| `gewi_negative` | GEWI | negative-polarity GEWI component |
| `gewi_v3` | GEWI | follower-weighted visibility sensitivity measure |
| `gewi_direct` | derived | tract met the direct-observation threshold |
| `geometry` | census tracts | polygon used to obtain projected centroid coordinates |

The notebook reports only schemas, counts, diagnostics, coefficients, and aggregate figures. It never displays raw rows.

In [ ]:
ALIASES = {
    "tract_id": ["tract_id", "CD_SETOR", "cd_setor", "code_censo", "CD_GEOCODI"],
    "total_visits": ["total_visits", "visits"],
    "total_unique_visitors": ["total_unique_visitors", "unique"],
    "visit_frequency": ["visit_frequency", "frequency"],
    "total_repeat_visitors": ["total_repeat_visitors", "repeat_visitors"],
    "total_new_visitors": ["total_new_visitors", "new_visitors"],
    "avg_dwell_time_mins": ["avg_dwell_time_mins", "dwell_time_mins"],
    "total_sales": ["total_sales", "total_sales_sum", "transaction_value"],
    "message_count": ["message_count", "n_messages", "tweet_count", "volume_tweets"],
    "gewi_v1": ["gewi_v1", "GEWI_v1", "GEWI", "gewi"],
    "gewi_positive": ["gewi_positive", "GEWI_v2_pos", "gewi_v2_pos", "GEWI_pos"],
    "gewi_negative": ["gewi_negative", "GEWI_v2_neg", "gewi_v2_neg", "GEWI_neg"],
    "gewi_v3": ["gewi_v3", "GEWI_v3", "gewi_visibility_weighted"],
}

MOBILITY_REQUIRED = [
    "tract_id", "total_visits", "total_unique_visitors",
    "total_repeat_visitors", "total_new_visitors", "avg_dwell_time_mins",
]
SALES_REQUIRED = ["tract_id", "total_sales"]
GEWI_REQUIRED = [
    "tract_id", "message_count", "gewi_v1", "gewi_positive", "gewi_negative",
]

In [ ]:
def read_tabular(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Required authorized input not found: {path.relative_to(CHAPTER_DIR)}"
        )
    suffixes = "".join(path.suffixes).lower()
    if suffixes.endswith((".csv", ".csv.gz")):
        return pd.read_csv(path, low_memory=False)
    if suffixes.endswith(".parquet"):
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported tabular format: {path.name}")


def normalize_tract_id(series):
    values = series.astype("string").str.replace(r"\.0$", "", regex=True).str.strip()
    invalid = values.isna() | ~values.str.fullmatch(r"\d{1,15}", na=False)
    if invalid.any():
        raise ValueError(f"Invalid census-tract identifiers: {int(invalid.sum())}")
    return values.str.zfill(15)


def canonicalize_columns(frame, required, optional=()):
    rename = {}
    for canonical in [*required, *optional]:
        candidates = [name for name in ALIASES[canonical] if name in frame.columns]
        if candidates:
            rename[candidates[0]] = canonical
        elif canonical in required:
            raise KeyError(
                f"Missing '{canonical}'. Accepted names: {ALIASES[canonical]}"
            )
    out = frame.rename(columns=rename).copy()
    out["tract_id"] = normalize_tract_id(out["tract_id"])
    keep = [name for name in [*required, *optional] if name in out.columns]
    return out[keep]


def assert_one_row_per_tract(frame, label):
    duplicates = frame["tract_id"].duplicated().sum()
    if duplicates:
        raise ValueError(f"{label} contains {duplicates} duplicate tract identifiers.")


def input_summary(**frames):
    rows = []
    for label, frame in frames.items():
        rows.append({
            "input": label,
            "rows": len(frame),
            "tracts": frame["tract_id"].nunique(),
            "variables": len(frame.columns),
        })
    return pd.DataFrame(rows)

## 3. Load the Authorized Tract-Level Inputs

The following cell loads only aggregated or derived files. It does not print any empirical record, message, transaction, or geographic identifier.

In [ ]:
mobility = canonicalize_columns(
    read_tabular(INPUTS["mobility"]),
    required=MOBILITY_REQUIRED,
    optional=["visit_frequency"],
)
sales = canonicalize_columns(
    read_tabular(INPUTS["sales"]),
    required=SALES_REQUIRED,
)
gewi_observed = canonicalize_columns(
    read_tabular(INPUTS["gewi"]),
    required=GEWI_REQUIRED,
    optional=["gewi_v3"],
)

for label, frame in {
    "mobility": mobility,
    "sales": sales,
    "GEWI": gewi_observed,
}.items():
    assert_one_row_per_tract(frame, label)

if "visit_frequency" not in mobility.columns:
    mobility["visit_frequency"] = np.where(
        mobility["total_unique_visitors"] > 0,
        mobility["total_visits"] / mobility["total_unique_visitors"],
        np.nan,
    )

input_summary(mobility=mobility, sales=sales, gewi=gewi_observed)

## 4. Census-Tract Coordinates and Spatial Support

Centroids are calculated from official tract polygons after projection to a metric coordinate reference system. The analysis does not derive coordinates from the digits of a census-tract identifier.

In [ ]:
if not INPUTS["tracts"].exists():
    raise FileNotFoundError(
        f"Required authorized input not found: {INPUTS['tracts'].relative_to(CHAPTER_DIR)}"
    )

geometry_read_options = {} if CENSUS_LAYER is None else {"layer": CENSUS_LAYER}
tracts_raw = gpd.read_file(INPUTS["tracts"], **geometry_read_options)
tract_id_candidates = [name for name in ALIASES["tract_id"] if name in tracts_raw.columns]
if not tract_id_candidates:
    raise KeyError(
        f"Missing tract identifier in geometry input. Accepted names: {ALIASES['tract_id']}"
    )
tracts = tracts_raw[[tract_id_candidates[0], "geometry"]].rename(
    columns={tract_id_candidates[0]: "tract_id"}
)
tracts = gpd.GeoDataFrame(tracts, geometry="geometry", crs=tracts_raw.crs)
tracts["tract_id"] = normalize_tract_id(tracts["tract_id"])

if tracts.crs is None:
    raise ValueError("The census-tract geometry file must declare its source CRS.")
if tracts["tract_id"].duplicated().any():
    raise ValueError("The geometry input contains duplicate tract identifiers.")
if tracts.geometry.isna().any() or tracts.geometry.is_empty.any():
    raise ValueError("The geometry input contains missing or empty polygons.")

tracts_projected = tracts.to_crs(PROJECTED_CRS)
centroids = tracts_projected.geometry.centroid
coordinates = pd.DataFrame({
    "tract_id": tracts_projected["tract_id"].to_numpy(),
    "centroid_x_m": centroids.x.to_numpy(),
    "centroid_y_m": centroids.y.to_numpy(),
})
assert_one_row_per_tract(coordinates, "coordinates")

pd.DataFrame({
    "geometry_rows": [len(tracts_projected)],
    "unique_tracts": [coordinates["tract_id"].nunique()],
    "projected_crs": [PROJECTED_CRS],
})

## 5. GEWI Direct Observation and IDW Coverage Expansion

Direct GEWI requires at least five relevant geolocated messages within a tract. For a tract below this threshold, the notebook uses inverse-distance weighting from directly observed tract centroids located within 2 kilometers. Direct values are always preserved, and expanded values remain explicitly flagged as interpolated.

In [ ]:
GEWI_COLUMNS = ["gewi_v1", "gewi_positive", "gewi_negative"]
if "gewi_v3" in gewi_observed.columns:
    GEWI_COLUMNS.append("gewi_v3")

gewi_observed["gewi_direct"] = (
    (gewi_observed["message_count"] >= DIRECT_MESSAGE_THRESHOLD)
    & gewi_observed["gewi_v1"].notna()
)

# A missing polarity component in a directly observed tract means that no
# retained message contributed to that polarity, so its component is zero.
for polarity in ["gewi_positive", "gewi_negative"]:
    mask = gewi_observed["gewi_direct"] & gewi_observed[polarity].isna()
    gewi_observed.loc[mask, polarity] = 0.0

donors = (
    gewi_observed.loc[gewi_observed["gewi_direct"]]
    .merge(coordinates, on="tract_id", how="inner", validate="one_to_one")
)
if donors.empty:
    raise ValueError("No directly observed GEWI donors are available.")

mobility_sales = (
    mobility.merge(sales, on="tract_id", how="inner", validate="one_to_one")
    .merge(coordinates, on="tract_id", how="inner", validate="one_to_one")
)

In [ ]:
def idw_expand(targets, observed_donors, value_columns, radius_m, power=1.0):
    target_xy = targets[["centroid_x_m", "centroid_y_m"]].to_numpy(float)
    donor_xy = observed_donors[["centroid_x_m", "centroid_y_m"]].to_numpy(float)

    search = NearestNeighbors(radius=radius_m, algorithm="ball_tree")
    search.fit(donor_xy)
    distances, indices = search.radius_neighbors(target_xy, return_distance=True)

    donor_values = observed_donors[value_columns].to_numpy(float)
    expanded = np.full((len(targets), len(value_columns)), np.nan, dtype=float)
    donor_count = np.zeros(len(targets), dtype=int)
    nearest_distance = np.full(len(targets), np.nan, dtype=float)

    for row, (row_distances, row_indices) in enumerate(zip(distances, indices)):
        if len(row_indices) == 0:
            continue
        valid = np.isfinite(row_distances)
        row_distances = row_distances[valid]
        row_indices = row_indices[valid]
        if len(row_indices) == 0:
            continue
        weights = 1.0 / np.maximum(row_distances, 1e-9) ** power
        values = donor_values[row_indices]
        for column in range(values.shape[1]):
            available = np.isfinite(values[:, column])
            if available.any():
                expanded[row, column] = np.average(
                    values[available, column], weights=weights[available]
                )
        donor_count[row] = len(row_indices)
        nearest_distance[row] = row_distances.min()

    result = targets[["tract_id"]].copy()
    for column, values in zip(value_columns, expanded.T):
        result[f"{column}_idw"] = values
    result["idw_donor_count"] = donor_count
    result["idw_nearest_distance_m"] = nearest_distance
    return result


expanded = idw_expand(
    mobility_sales,
    donors,
    value_columns=GEWI_COLUMNS,
    radius_m=IDW_RADIUS_METERS,
    power=IDW_POWER,
)

direct_fields = ["tract_id", "message_count", "gewi_direct", *GEWI_COLUMNS]
analysis = (
    mobility_sales
    .merge(gewi_observed[direct_fields], on="tract_id", how="left", validate="one_to_one")
    .merge(expanded, on="tract_id", how="left", validate="one_to_one")
)
analysis["gewi_direct"] = analysis["gewi_direct"].fillna(False).astype(bool)

for column in GEWI_COLUMNS:
    analysis[column] = analysis[column].where(
        analysis["gewi_direct"], analysis[f"{column}_idw"]
    )

analysis = analysis.dropna(
    subset=["total_sales", FOCAL_MOBILITY, "gewi_v1", "centroid_x_m", "centroid_y_m"]
).reset_index(drop=True)

sample_audit = pd.DataFrame({
    "integrated_tracts": [len(analysis)],
    "direct_GEWI_tracts": [int(analysis["gewi_direct"].sum())],
    "expanded_GEWI_tracts": [int((~analysis["gewi_direct"]).sum())],
})

if STRICT_REPLICATION:
    assert len(analysis) == 726, "The integrated sample does not match the dissertation."
    assert analysis["gewi_direct"].sum() == 96, "The direct GEWI subset does not match."

sample_audit

## 6. Analytical Transformations

The focal relationship is estimated in log-log form. Interaction terms retain the GEWI scale used in the dissertation, and the direct-observation indicator remains available for sensitivity analysis.

In [ ]:
analysis["log_sales"] = np.log1p(analysis["total_sales"])
analysis["log_mobility"] = np.log1p(analysis[FOCAL_MOBILITY])
analysis["mobility_x_gewi"] = analysis["log_mobility"] * analysis["gewi_v1"]
analysis["mobility_x_positive"] = analysis["log_mobility"] * analysis["gewi_positive"]
analysis["mobility_x_negative"] = analysis["log_mobility"] * analysis["gewi_negative"]

for column in [
    "total_visits", "total_unique_visitors", "total_repeat_visitors",
    "total_new_visitors", "avg_dwell_time_mins", "visit_frequency",
]:
    analysis[f"log_{column}"] = np.log1p(analysis[column].clip(lower=0))

analysis_schema = pd.DataFrame({
    "variable": analysis.columns,
    "dtype": analysis.dtypes.astype(str).to_numpy(),
    "missing": analysis.isna().sum().to_numpy(),
})
analysis_schema

## 7. Spatial Weights and Moran's I

The baseline spatial weights matrix connects each tract to its eight nearest tract centroids and is row-standardized. Moran's I is calculated for log-transformed localized sales. This diagnostic motivates spatial model choice but does not establish a behavioral spillover.

In [ ]:
def build_knn_weights(frame, k=K_NEIGHBORS):
    if frame["tract_id"].duplicated().any():
        raise ValueError("Spatial weights require one row per tract.")
    xy = frame[["centroid_x_m", "centroid_y_m"]].to_numpy(float)
    weights = KNN.from_array(xy, k=k, ids=frame["tract_id"].tolist())
    weights.transform = "r"
    if weights.islands:
        raise ValueError(f"Spatial weights contain islands: {weights.islands}")
    return weights


analysis = analysis.sort_values("tract_id").reset_index(drop=True)
W = build_knn_weights(analysis, k=K_NEIGHBORS)
moran_sales = Moran(analysis["log_sales"].to_numpy(), W, permutations=999)

pd.DataFrame({
    "n": [W.n],
    "k": [K_NEIGHBORS],
    "row_standardized": [W.transform == "R"],
    "morans_I": [moran_sales.I],
    "permutation_p": [moran_sales.p_sim],
})

## 8. Baseline Spatial Lag and Spatial Error Models

The coefficient on `log_mobility` is read as the tract-level mobility-sales elasticity. The spatial parameters account for spatial dependence in the outcome or residual process; they are not interpreted as behavioral or commercial spillovers.

In [ ]:
def fit_spatial_pair(frame, x_columns, k=K_NEIGHBORS, label="model"):
    required = ["tract_id", "log_sales", "centroid_x_m", "centroid_y_m", *x_columns]
    work = frame.dropna(subset=required).sort_values("tract_id").reset_index(drop=True)
    weights = build_knn_weights(work, k=min(k, len(work) - 1))
    y = work[["log_sales"]].to_numpy(float)
    X = work[x_columns].to_numpy(float)

    lag = ML_Lag(
        y, X, w=weights,
        name_y="log_sales", name_x=x_columns, name_ds=label,
        method="full",
    )
    error = ML_Error(
        y, X, w=weights,
        name_y="log_sales", name_x=x_columns, name_ds=label,
        method="full",
    )
    return {"lag": lag, "error": error, "weights": weights, "data": work}


def coefficient_table(model, model_name):
    estimates = np.asarray(model.betas).reshape(-1)
    standard_errors = np.asarray(model.std_err).reshape(-1)
    z_stats = list(model.z_stat)
    names = list(model.name_x)
    if len(names) < len(estimates):
        names += [f"spatial_parameter_{i}" for i in range(len(estimates) - len(names))]
    return pd.DataFrame({
        "model": model_name,
        "term": names[:len(estimates)],
        "estimate": estimates,
        "std_error": standard_errors,
        "z": [item[0] for item in z_stats],
        "p_value": [item[1] for item in z_stats],
        "n": model.n,
    })


baseline = fit_spatial_pair(analysis, ["log_mobility"], label="baseline")
baseline_results = pd.concat([
    coefficient_table(baseline["lag"], "Spatial Lag Model"),
    coefficient_table(baseline["error"], "Spatial Error Model"),
], ignore_index=True)
baseline_results

In [ ]:
REFERENCE_BASELINE = pd.DataFrame([
    {"model": "Spatial Lag Model", "term": "log_mobility", "estimate": 0.3275,
     "std_error": 0.0889, "p_value": 0.0002, "n": 726},
    {"model": "Spatial Error Model", "term": "log_mobility", "estimate": 0.3657,
     "std_error": 0.0909, "p_value": 0.0001, "n": 726},
])


def compare_with_reference(calculated, reference, keys=("model", "term")):
    columns = [*keys, "estimate", "std_error", "p_value", "n"]
    left = calculated[calculated["term"].isin(reference["term"])][columns]
    out = reference.merge(left, on=list(keys), how="left", suffixes=("_reported", "_calculated"))
    for metric in ["estimate", "std_error", "p_value"]:
        out[f"{metric}_difference"] = out[f"{metric}_calculated"] - out[f"{metric}_reported"]
    return out


baseline_reference_check = compare_with_reference(baseline_results, REFERENCE_BASELINE)
baseline_reference_check

## 9. Baseline GEWI Interaction

The interaction between log mobility and baseline GEWI examines whether the mobility-sales association differs with the place-based digital sentiment observed around a tract. Both spatial specifications are estimated so that the sign and magnitude can be inspected across model choices.

In [ ]:
interaction_columns = ["log_mobility", "gewi_v1", "mobility_x_gewi"]
gewi_interaction = fit_spatial_pair(
    analysis, interaction_columns, label="baseline_GEWI_interaction"
)
gewi_interaction_results = pd.concat([
    coefficient_table(gewi_interaction["lag"], "Spatial Lag Model"),
    coefficient_table(gewi_interaction["error"], "Spatial Error Model"),
], ignore_index=True)
gewi_interaction_results

In [ ]:
REFERENCE_GEWI_INTERACTION = pd.DataFrame([
    {"term": "mobility_x_gewi", "reported_estimate": 0.127, "reported_p_value": 0.0034}
])

reported_interaction_comparison = (
    gewi_interaction_results.loc[
        gewi_interaction_results["term"] == "mobility_x_gewi",
        ["model", "term", "estimate", "std_error", "p_value", "n"],
    ]
    .merge(REFERENCE_GEWI_INTERACTION, on="term", how="left")
)
reported_interaction_comparison

## 10. Polarity-Specific GEWI Interactions

Positive and negative GEWI enter the same spatial specification with separate main and interaction terms. The difference between the reported positive and negative interaction estimates is interpreted descriptively rather than as a separately tested coefficient contrast.

In [ ]:
polarity_columns = [
    "log_mobility", "gewi_positive", "gewi_negative",
    "mobility_x_positive", "mobility_x_negative",
]
polarity = fit_spatial_pair(analysis, polarity_columns, label="polarity_GEWI_interactions")
polarity_results = pd.concat([
    coefficient_table(polarity["lag"], "Spatial Lag Model"),
    coefficient_table(polarity["error"], "Spatial Error Model"),
], ignore_index=True)
polarity_results

In [ ]:
REFERENCE_POLARITY = pd.DataFrame([
    {"term": "mobility_x_positive", "reported_estimate": 0.072, "reported_p_value": 0.012},
    {"term": "mobility_x_negative", "reported_estimate": -0.183, "reported_p_value": 0.004},
])

reported_polarity_comparison = (
    polarity_results.loc[
        polarity_results["term"].isin(REFERENCE_POLARITY["term"]),
        ["model", "term", "estimate", "std_error", "p_value", "n"],
    ]
    .merge(REFERENCE_POLARITY, on="term", how="left")
)
reported_polarity_comparison

## 11. Directly Observed GEWI Sensitivity

The non-interpolated subset retains only tracts whose GEWI was calculated from at least five messages observed inside the tract. This comparison evaluates sensitivity to coverage expansion; it does not make the 96-tract subset representative of Brazilian local markets.

In [ ]:
direct_analysis = analysis.loc[analysis["gewi_direct"]].copy()
if STRICT_REPLICATION:
    assert len(direct_analysis) == 96

direct_interaction = fit_spatial_pair(
    direct_analysis, interaction_columns, label="direct_GEWI_interaction"
)
direct_polarity = fit_spatial_pair(
    direct_analysis, polarity_columns, label="direct_GEWI_polarity"
)

direct_sensitivity_results = pd.concat([
    coefficient_table(direct_interaction["lag"], "Direct subset: Spatial Lag"),
    coefficient_table(direct_interaction["error"], "Direct subset: Spatial Error"),
    coefficient_table(direct_polarity["lag"], "Direct subset polarity: Spatial Lag"),
    coefficient_table(direct_polarity["error"], "Direct subset polarity: Spatial Error"),
], ignore_index=True)
direct_sensitivity_results

## 12. Leave-One-Out Evaluation of the GEWI Coverage Rule

Each directly observed tract is removed in turn and reconstructed from the remaining directly observed tracts within 2 kilometers. The resulting R2 evaluates the selected spatial expansion rule, not the equivalence of observed and interpolated discourse.

In [ ]:
def leave_one_out_idw(observed, value_column, radius_m=IDW_RADIUS_METERS, power=IDW_POWER):
    observed = observed.dropna(
        subset=[value_column, "centroid_x_m", "centroid_y_m"]
    ).reset_index(drop=True)
    xy = observed[["centroid_x_m", "centroid_y_m"]].to_numpy(float)
    values = observed[value_column].to_numpy(float)
    search = NearestNeighbors(radius=radius_m, algorithm="ball_tree").fit(xy)
    distances, indices = search.radius_neighbors(xy, return_distance=True)
    predictions = np.full(len(observed), np.nan)

    for row, (row_distances, row_indices) in enumerate(zip(distances, indices)):
        keep = row_indices != row
        row_distances = row_distances[keep]
        row_indices = row_indices[keep]
        if len(row_indices) == 0:
            continue
        weights = 1.0 / np.maximum(row_distances, 1e-9) ** power
        predictions[row] = np.average(values[row_indices], weights=weights)

    valid = np.isfinite(predictions)
    return {
        "observed_tracts": len(observed),
        "reconstructed_tracts": int(valid.sum()),
        "coverage_share": float(valid.mean()),
        "r2": r2_score(values[valid], predictions[valid]),
        "rmse": mean_squared_error(values[valid], predictions[valid]) ** 0.5,
    }


loo_results = leave_one_out_idw(direct_analysis, "gewi_v1")
pd.DataFrame([loo_results]).assign(dissertation_reference_r2=0.794)

## 13. Alternative Spatial Weights, GEWI v3, and Collinearity

These cells evaluate whether the main sign and magnitude are sensitive to reasonable neighborhood definitions, whether follower-weighted GEWI changes the substantive pattern, and whether the interaction specification contains severe linear dependence.

In [ ]:
alternative_weight_rows = []
for k in [4, 8, 12]:
    fitted = fit_spatial_pair(analysis, interaction_columns, k=k, label=f"GEWI_k{k}")
    for model_key, model_name in [("lag", "Spatial Lag"), ("error", "Spatial Error")]:
        table = coefficient_table(fitted[model_key], model_name)
        row = table.loc[table["term"] == "mobility_x_gewi"].iloc[0]
        alternative_weight_rows.append({
            "k": k,
            "model": model_name,
            "interaction_estimate": row["estimate"],
            "std_error": row["std_error"],
            "p_value": row["p_value"],
            "n": row["n"],
        })

alternative_weights_results = pd.DataFrame(alternative_weight_rows)
alternative_weights_results

In [ ]:
if "gewi_v3" in analysis.columns and analysis["gewi_v3"].notna().any():
    analysis["mobility_x_gewi_v3"] = analysis["log_mobility"] * analysis["gewi_v3"]
    gewi_v3_models = fit_spatial_pair(
        analysis,
        ["log_mobility", "gewi_v3", "mobility_x_gewi_v3"],
        label="GEWI_v3_sensitivity",
    )
    gewi_v3_results = pd.concat([
        coefficient_table(gewi_v3_models["lag"], "GEWI v3: Spatial Lag"),
        coefficient_table(gewi_v3_models["error"], "GEWI v3: Spatial Error"),
    ], ignore_index=True)
else:
    gewi_v3_results = pd.DataFrame({
        "status": ["GEWI v3 was not supplied in the authorized derived input."]
    })

gewi_v3_results

In [ ]:
vif_columns = polarity_columns
vif_input = sm.add_constant(analysis[vif_columns].dropna(), has_constant="add")
vif_results = pd.DataFrame({
    "term": vif_input.columns,
    "VIF": [
        variance_inflation_factor(vif_input.to_numpy(), i)
        for i in range(vif_input.shape[1])
    ],
})
vif_results

## 14. Flexible-Model Support Diagnostics (Figures B1 and B2)

For each focal mobility measure, the upper quartile defines a descriptive high-mobility group. A logistic model built from the remaining covariates produces group-membership scores used only to examine empirical overlap. These scores are not treatment propensities and do not establish causal identification.

In [ ]:
def support_diagnostic(frame, focal_column, output_stem):
    candidate_covariates = [
        "total_visits", "total_unique_visitors", "total_repeat_visitors",
        "total_new_visitors", "avg_dwell_time_mins",
        "gewi_positive", "gewi_negative",
    ]
    covariates = [column for column in candidate_covariates if column != focal_column]
    work = frame[[focal_column, *covariates]].dropna().copy()
    threshold = work[focal_column].quantile(0.75)
    work["high_mobility_group"] = (work[focal_column] >= threshold).astype(int)

    model = Pipeline([
        ("scale", StandardScaler()),
        ("logit", LogisticRegression(max_iter=5_000, random_state=RANDOM_STATE)),
    ])
    model.fit(work[covariates], work["high_mobility_group"])
    work["support_score"] = model.predict_proba(work[covariates])[:, 1]

    high = work.loc[work["high_mobility_group"] == 1, "support_score"]
    comparison = work.loc[work["high_mobility_group"] == 0, "support_score"]
    lower = max(high.min(), comparison.min())
    upper = min(high.max(), comparison.max())

    figure, axis = plt.subplots(figsize=(8, 5))
    bins = np.linspace(0, 1, 31)
    axis.hist(comparison, bins=bins, density=True, alpha=0.60, label="Remaining tracts")
    axis.hist(high, bins=bins, density=True, alpha=0.60, label="Upper mobility quartile")
    axis.axvline(lower, color="black", linestyle="--", linewidth=1)
    axis.axvline(upper, color="black", linestyle="--", linewidth=1)
    axis.set(xlabel="Estimated group-membership score", ylabel="Density")
    axis.legend(frameon=True)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / f"{output_stem}.png", dpi=300, bbox_inches="tight")

    return pd.DataFrame([{
        "focal_measure": focal_column,
        "n": len(work),
        "high_group_n": int(work["high_mobility_group"].sum()),
        "common_support_lower": lower,
        "common_support_upper": upper,
        "share_within_common_support": work["support_score"].between(lower, upper).mean(),
    }])


support_unique = support_diagnostic(
    analysis, "total_unique_visitors", "figure_B1_support_unique_visitors"
)
support_visits = support_diagnostic(
    analysis, "total_visits", "figure_B2_support_total_visits"
)
pd.concat([support_unique, support_visits], ignore_index=True)

## 15. Flexible Tree-Based Partitioning Diagnostic (Figure B3)

Chapter 3 uses a causal-forest architecture only as a flexible partitioning method. Consumer mobility was not assigned, and the design does not satisfy the assumptions needed for treatment-effect interpretation. The values produced below are labeled **conditional associations**, not CATEs or causal effects.

This cell requires the optional `econml` package. It uses the parameters retained from the working analysis: two random-forest nuisance models, 2,000 trees in the partitioning model, minimum leaf size of 5, maximum depth of 15, and random seed 42.

In [ ]:
try:
    from econml.dml import CausalForestDML
except ImportError as exc:
    raise ImportError(
        "Install the econml version recorded for the authorized analysis environment "
        "before running the optional flexible-partitioning diagnostic."
    ) from exc

forest_frame = analysis.dropna(
    subset=["log_sales", "log_mobility", "gewi_positive", "gewi_negative"]
).copy()
Y_forest = forest_frame["log_sales"].to_numpy()
T_forest = forest_frame["log_mobility"].to_numpy()
X_forest = forest_frame[["gewi_positive", "gewi_negative"]].to_numpy()
X_forest = StandardScaler().fit_transform(X_forest)

nuisance_y = RandomForestRegressor(
    n_estimators=200, max_depth=8, n_jobs=-1, random_state=RANDOM_STATE
)
nuisance_t = RandomForestRegressor(
    n_estimators=200, max_depth=8, n_jobs=-1, random_state=RANDOM_STATE
)
partition_model = CausalForestDML(
    model_y=nuisance_y,
    model_t=nuisance_t,
    n_estimators=2_000,
    min_samples_leaf=5,
    max_depth=15,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
partition_model.fit(Y_forest, T_forest, X=X_forest)

forest_frame["conditional_mobility_sales_association"] = partition_model.effect(X_forest)

figure, axis = plt.subplots(figsize=(8, 5))
sns.histplot(
    forest_frame["conditional_mobility_sales_association"],
    bins=30, kde=True, ax=axis,
)
axis.axvline(
    forest_frame["conditional_mobility_sales_association"].mean(),
    color="black", linestyle="--", linewidth=1.2,
)
axis.set(
    xlabel="Model-derived conditional mobility-sales association",
    ylabel="Census tracts",
)
figure.tight_layout()
figure.savefig(
    FIGURE_DIR / "figure_B3_conditional_associations.png",
    dpi=300, bbox_inches="tight",
)

forest_frame["conditional_mobility_sales_association"].describe()

## 16. Random Forest and XGBoost Predictive Diagnostics (Table B2)

These models use the mobility-sales overlap available before GEWI coverage is imposed, because their purpose is to ask whether aggregated mobility measures alone reveal a stronger nonlinear predictive structure. Both models use the same predictors, log-sales outcome, 80/20 split, and five cross-validation folds. Their performance does not validate the spatial models or establish geographic transferability.

In [ ]:
FLEXIBLE_PREDICTORS = [
    "total_unique_visitors", "total_visits", "total_repeat_visitors",
    "total_new_visitors", "avg_dwell_time_mins",
]
predictive_frame = mobility_sales.dropna(
    subset=["total_sales", *FLEXIBLE_PREDICTORS]
).copy()
X_predictive = predictive_frame[FLEXIBLE_PREDICTORS].to_numpy()
y_predictive = np.log1p(predictive_frame["total_sales"].to_numpy())

X_train, X_test, y_train, y_test = train_test_split(
    X_predictive,
    y_predictive,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    ),
    "XGBoost": XGBRegressor(
        n_estimators=800,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        objective="reg:squarederror",
    ),
}

folds = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
predictive_rows = []
for model_name, model in models.items():
    model.fit(X_train, y_train)
    test_prediction = model.predict(X_test)
    cv = cross_validate(
        model,
        X_predictive,
        y_predictive,
        scoring=("r2", "neg_root_mean_squared_error"),
        cv=folds,
        n_jobs=-1,
    )
    predictive_rows.append({
        "model": model_name,
        "predictors": "Aggregated mobility measures",
        "analytical_n": len(predictive_frame),
        "test_r2": r2_score(y_test, test_prediction),
        "test_rmse": mean_squared_error(y_test, test_prediction) ** 0.5,
        "cv_r2_mean": cv["test_r2"].mean(),
        "cv_r2_sd": cv["test_r2"].std(),
        "cv_rmse_mean": (-cv["test_neg_root_mean_squared_error"]).mean(),
        "cv_rmse_sd": (-cv["test_neg_root_mean_squared_error"]).std(),
    })

predictive_results = pd.DataFrame(predictive_rows)
predictive_results

In [ ]:
REFERENCE_FLEXIBLE_MODELS = pd.DataFrame([
    {
        "model": "Random Forest", "test_r2_reported": 0.024,
        "test_rmse_reported": 2.215, "cv_r2_mean_reported": 0.005,
        "cv_r2_sd_reported": 0.015, "cv_rmse_mean_reported": 2.197,
        "cv_rmse_sd_reported": 0.058,
    },
    {
        "model": "XGBoost", "test_r2_reported": -0.058,
        "test_rmse_reported": 2.306, "cv_r2_mean_reported": -0.082,
        "cv_r2_sd_reported": 0.030, "cv_rmse_mean_reported": 2.291,
        "cv_rmse_sd_reported": 0.042,
    },
])

flexible_reference_check = REFERENCE_FLEXIBLE_MODELS.merge(
    predictive_results, on="model", how="left"
)
flexible_reference_check

## 17. Private Output Inventory

The workflow can save aggregate diagnostic figures and tables under `data/private/analysis/`. These files may contain derived tract-level values and must be reviewed and cleared before any public commit. The public repository should contain this notebook with empty outputs, the data dictionaries, and the documentation, but not the restricted inputs or locally generated analytical files.

In [ ]:
public_release_check = pd.DataFrame({
    "check": [
        "Notebook outputs cleared before commit",
        "Restricted inputs excluded",
        "Private analytical outputs excluded",
        "No absolute paths or credentials",
        "No individual mobility, message, or transaction records",
    ],
    "required_status": ["Yes"] * 5,
})
public_release_check